In [2]:
import os, json
import pandas as pd
import numpy as np
import torch
from importlib.metadata import version

KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'
RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_v2'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'PyKEEN  : {version("pykeen")}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
print(f'Output  : {RESULTS_DIR}')


PyKEEN  : 1.11.1
PyTorch : 2.6.0+cu124
GPU     : True
GPU name: NVIDIA GeForce GTX 1650
Output  : D:\Memoire\LLMKGER\kge_results_v2


## Étape 1 — Préparer le dataset KGE avec follows


In [2]:
SAMPLE_FOLLOWS = 500_000  # sous-échantillon de follows

# ── 1. Charger triplets KG sémantique (hors follows) ──────────────────────
df_kg = pd.read_csv(
    os.path.join(KG_DIR, 'kg_train.tsv'),
    sep='\t', header=0, names=['head','relation','tail']
)
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()
print(f'Triplets KG sémantique : {len(df_kg_sem):,}')

# ── 2. Charger interactions follows (train seulement) ─────────────────────
df_follows = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_train.tsv'),
    sep='\t', header=0, names=['head','relation','tail']
)
print(f'Follows train total    : {len(df_follows):,}')

# ── 3. Sous-échantillon de follows ────────────────────────────────────────
df_follows_sample = df_follows.sample(n=SAMPLE_FOLLOWS, random_state=42)
print(f'Follows sous-échantillon : {len(df_follows_sample):,}')

# ── 4. Combiner KG sémantique + follows ───────────────────────────────────
df_combined = pd.concat([df_kg_sem, df_follows_sample], ignore_index=True)
print(f'\nDataset combiné total  : {len(df_combined):,}')
print(f'\nDistribution relations :')
print(df_combined['relation'].value_counts())

# ── 5. Vérifier que users et cours sont bien présents ─────────────────────
all_entities = set(df_combined['head'].unique()) | set(df_combined['tail'].unique())
users = {e for e in all_entities if e.startswith('U_')}
courses = {e for e in all_entities if e.startswith('C_')}
print(f'\nEntités totales : {len(all_entities):,}')
print(f'Users           : {len(users):,}')
print(f'Cours           : {len(courses):,}')


Triplets KG sémantique : 648,349
Follows train total    : 5,044,890
Follows sous-échantillon : 500,000

Dataset combiné total  : 1,148,349

Distribution relations :
relation
follows            500000
has_concept        251334
belongs_to         211691
watches            112038
prerequisite_of     59751
taught_by           10742
offered_by           2793
Name: count, dtype: int64

Entités totales : 235,055
Users           : 104,677
Cours           : 2,822


In [3]:
from pykeen.triples import TriplesFactory

# Créer le TriplesFactory avec le dataset combiné
tf = TriplesFactory.from_labeled_triples(
    triples=df_combined[['head','relation','tail']].values,
    create_inverse_triples=False
)

# Split 85/7.5/7.5
train_tf, valid_tf, test_tf = tf.split([0.85, 0.075, 0.075], random_state=42)

print(f'Train : {train_tf.num_triples:,}')
print(f'Valid : {valid_tf.num_triples:,}')
print(f'Test  : {test_tf.num_triples:,}')
print(f'Entités   : {tf.num_entities:,}')
print(f'Relations : {tf.num_relations:,}')
print(f'\nRelations disponibles : {list(tf.relation_to_id.keys())}')


Train : 976,096
Valid : 86,126
Test  : 86,127
Entités   : 235,055
Relations : 7

Relations disponibles : ['belongs_to', 'follows', 'has_concept', 'offered_by', 'prerequisite_of', 'taught_by', 'watches']


## Étape 2 — Entraînement RotatE (avec follows)


In [4]:
from pykeen.pipeline import pipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print('Entraînement RotatE v2 (avec follows)...')

result = pipeline(
    training         = train_tf,
    validation       = valid_tf,
    testing          = test_tf,
    model            = 'RotatE',
    model_kwargs     = {'embedding_dim': 128},
    loss             = 'NSSALoss',
    optimizer        = 'Adam',
    optimizer_kwargs = {'lr': 0.001},
    training_kwargs  = {
        'num_epochs' : 100,
        'batch_size' : 1024,
    },
    evaluator_kwargs = {
        'filtered'   : True,
        'batch_size' : 32,
    },
    random_seed      = 42,
    device           = DEVICE,
)

# Métriques
metrics = result.metric_results.to_dict()
m = metrics['both']['realistic']
mrr = m['inverse_harmonic_mean_rank']
h1  = m['hits_at_1']
h3  = m['hits_at_3']
h10 = m['hits_at_10']

print(f'\n=== RotatE v2 (avec follows) ===')
print(f'MRR    : {mrr:.4f}')
print(f'Hit@1  : {h1:.4f}')
print(f'Hit@3  : {h3:.4f}')
print(f'Hit@10 : {h10:.4f}')

# Sauvegarder
result.save_to_directory(os.path.join(RESULTS_DIR, 'RotatE_v2'))
print(f'\nModèle sauvegardé ✅')
  

Device : cuda
Entraînement RotatE v2 (avec follows)...


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/954 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/954 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/954 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/954 [00:00<?, ?batch/s]

... (97 outputs left out)

INFO:pykeen.evaluation.evaluator:Evaluation took 9206.50s seconds



=== RotatE v2 (avec follows) ===
MRR    : 0.0940
Hit@1  : 0.0588
Hit@3  : 0.0978
Hit@10 : 0.1598


INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=235055, num_relations=7, create_inverse_triples=False, num_triples=976096) to file:///D:/Memoire/LLMKGER/kge_results_v2/RotatE_v2/training_triples
INFO:pykeen.pipeline.api:Saved to directory: D:\Memoire\LLMKGER\kge_results_v2\RotatE_v2



Modèle sauvegardé ✅


## Étape 3 — Extraction S_KG correct


In [5]:
import torch.nn.functional as F

# Charger le modèle RotatE v2
trained_model = torch.load(
    os.path.join(RESULTS_DIR, 'RotatE_v2', 'trained_model.pkl'),
    map_location='cpu',
    weights_only=False
)
trained_model.eval()
print('RotatE v2 chargé ✅')

# Charger entity2id et relation2id
training_path = os.path.join(RESULTS_DIR, 'RotatE_v2', 'training_triples')
entity2id_df   = pd.read_csv(
    os.path.join(training_path, 'entity_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)
relation2id_df = pd.read_csv(
    os.path.join(training_path, 'relation_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)

entity2id   = dict(zip(entity2id_df['label'],   entity2id_df['id']))
relation2id = dict(zip(relation2id_df['label'], relation2id_df['id']))

print(f'Entités   : {len(entity2id):,}')
print(f'Relations : {list(relation2id.keys())}')

# Vérifier que follows et users sont présents
follows_id = relation2id.get('follows', -1)
print(f'\nID relation follows : {follows_id}')

# Test user et cours
test_user   = 'U_12868687'
test_course = 'C_934517'
print(f'User   {test_user}   in KG : {test_user in entity2id}')
print(f'Course {test_course} in KG : {test_course in entity2id}')


RotatE v2 chargé ✅
Entités   : 235,055
Relations : ['belongs_to', 'follows', 'has_concept', 'offered_by', 'prerequisite_of', 'taught_by', 'watches']

ID relation follows : 1
User   U_12868687   in KG : True
Course C_934517 in KG : True


In [6]:
import pandas as pd
import torch
import os

KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'
RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_v2'

# ── Recharger le modèle RotatE v2 ─────────────────────────────────────────
trained_model = torch.load(
    os.path.join(RESULTS_DIR, 'RotatE_v2', 'trained_model.pkl'),
    map_location='cpu',
    weights_only=False
)
trained_model.eval()
print('RotatE v2 rechargé ✅')

# ── Recharger entity2id et relation2id ────────────────────────────────────
training_path = os.path.join(RESULTS_DIR, 'RotatE_v2', 'training_triples')
entity2id_df  = pd.read_csv(
    os.path.join(training_path, 'entity_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)
relation2id_df = pd.read_csv(
    os.path.join(training_path, 'relation_to_id.tsv.gz'),
    sep='\t', compression='gzip'
)
entity2id   = dict(zip(entity2id_df['label'], entity2id_df['id']))
relation2id = dict(zip(relation2id_df['label'], relation2id_df['id']))
follows_id  = relation2id.get('follows', -1)
print(f'entity2id  : {len(entity2id):,} ✅')
print(f'follows_id : {follows_id} ✅')

# ── Fonction S_KG direct ──────────────────────────────────────────────────
def compute_s_kg_direct(user_id, course_id):
    if user_id not in entity2id or course_id not in entity2id:
        return 0.0
    if follows_id == -1:
        return 0.0
    h_idx = torch.tensor([[entity2id[user_id]]])
    r_idx = torch.tensor([[follows_id]])
    t_idx = torch.tensor([[entity2id[course_id]]])
    hrt   = torch.cat([h_idx, r_idx, t_idx], dim=1)
    with torch.no_grad():
        score = trained_model.score_hrt(hrt).item()
    return float(score)

# ── Test sur 5 paires ─────────────────────────────────────────────────────
interactions_train = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_train.tsv'),
    sep='\t', header=0, names=['user_id','relation','course_id']
)

print('\nTest S_KG direct sur 5 paires :')
for i in range(5):
    uid   = interactions_train['user_id'].iloc[i]
    cid   = interactions_train['course_id'].iloc[i]
    score = compute_s_kg_direct(uid, cid)
    print(f'  S_KG({uid}, {cid}) = {score:.4f}')

RotatE v2 rechargé ✅
entity2id  : 235,055 ✅
follows_id : 1 ✅

Test S_KG direct sur 5 paires :
  S_KG(U_12868687, C_934517) = -8.5613
  S_KG(U_12868687, C_694168) = -8.0528
  S_KG(U_12868687, C_814402) = -8.2766
  S_KG(U_12868687, C_696586) = -7.9975
  S_KG(U_12868687, C_947241) = -7.9700


In [7]:
from tqdm import tqdm

# Calculer S_KG pour le test set complet
interactions_test = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_test.tsv'),
    sep='\t', header=0, names=['user_id','relation','course_id']
)
print(f'Test set : {len(interactions_test):,} paires')

# Calculer en batch pour plus de vitesse
s_kg_records = []
not_in_kg = 0

for _, row in tqdm(interactions_test.iterrows(), total=len(interactions_test), desc='S_KG'):
    uid = row['user_id']
    cid = row['course_id']
    score = compute_s_kg_direct(uid, cid)
    if score == 0.0 and (uid not in entity2id or cid not in entity2id):
        not_in_kg += 1
    s_kg_records.append({'user_id': uid, 'course_id': cid, 'S_KG': score})

df_skg = pd.DataFrame(s_kg_records)

print(f'\nS_KG calculé pour {len(df_skg):,} paires')
print(f'Paires hors KG (score=0) : {not_in_kg:,}')
print(f'\nStatistiques S_KG :')
print(df_skg['S_KG'].describe())

# Sauvegarder
skg_path = os.path.join(RESULTS_DIR, 'S_KG_scores_final.csv')
df_skg.to_csv(skg_path, index=False)
print(f'\nS_KG sauvegardé → {skg_path} ✅')


Test set : 106,735 paires


S_KG: 100%|██████████| 106735/106735 [02:48<00:00, 632.65it/s] 



S_KG calculé pour 106,735 paires
Paires hors KG (score=0) : 2,058

Statistiques S_KG :
count    106735.000000
mean         -9.397140
std           1.972874
min         -19.725307
25%         -10.310526
50%          -9.302847
75%          -8.512312
max           0.000000
Name: S_KG, dtype: float64

S_KG sauvegardé → D:\Memoire\LLMKGER\kge_results_v2\S_KG_scores_final.csv ✅
